GenAI: I used Claude and HuggingFaceAI to learn how to download a set of npz files from HuggingFaceHub.

In [28]:
from datasets import load_dataset
from huggingface_hub import login
from huggingface_hub import hf_hub_download
from datasets import Dataset, concatenate_datasets
from transformers import pipeline
import numpy as np
from huggingface_hub import snapshot_download
import glob

In [7]:
login()

In [43]:
# sing_coil = load_dataset("AUMLProject/fastmri-knee-singlecoil-rss")
def download_one_example():
  filepath1 = hf_hub_download(
      repo_id="AUMLProject/fastmri-knee-singlecoil-rss",
      filename="data/file1000001.npz",
      repo_type="dataset"
  )
  npz1 = np.load(filepath1)

  return npz1

In [44]:
download_one_example()

NpzFile '/root/.cache/huggingface/hub/datasets--AUMLProject--fastmri-knee-singlecoil-rss/snapshots/be54b608d0ea125707a5bd69d4879fb27677c3a4/data/file1000001.npz' with keys: images, filename, num_slices, height, width...

In [35]:
# TODO: Check if this downloads fully and concats the datasets correctly
def download_full_singlecoil_dataset():
  local_dir = snapshot_download(
      repo_id="AUMLProject/fastmri-knee-singlecoil-rss",
      repo_type="dataset",
      allow_patterns="*.npz",

  )

  datasets = []
  for filepath in sorted(glob.glob(f"{local_dir}/data/*.npz")):
      npz = np.load(filepath, allow_pickle=True)
      datasets.append(Dataset.from_dict({k: npz[k] for k in npz.files}))

  dataset = concatenate_datasets(datasets)

  return dataset

  print(local_dir)


Fetching 973 files:   0%|          | 0/973 [00:00<?, ?it/s]

KeyboardInterrupt: 

TypeError: iteration over a 0-d array

In [4]:


pipe = pipeline("image-text-to-text", model="google/medgemma-1.5-4b-it")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
pipe(text=messages)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Load model directly
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained("google/medgemma-1.5-4b-it")
model = AutoModelForImageTextToText.from_pretrained("google/medgemma-1.5-4b-it")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
inputs = processor.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))